In [1]:
!rm -r ~/.cache/huggingface/datasets


rm: cannot remove '/root/.cache/huggingface/datasets': No such file or directory


In [2]:
!pip install -U datasets huggingface_hub fsspec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 514.8/514.8 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 13.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.32.4
    Uninstalling huggingface-hub-0.32.4:
      Successfully uninstalled huggingface-hub-0.32.4
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3

In [3]:
# ========================================
# Homework 5 - Parte 1: Financial Phrasebank con FinBERT
# ========================================

# 1. Instalar bibliotecas necesarias
!pip install transformers datasets --quiet

# 2. Importar librerías
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import pipeline
import pandas as pd



### 3. Cargar el dataset Financial Phrasebank

In [4]:
dataset = load_dataset("financial_phrasebank", "sentences_allagree")
df = pd.DataFrame(dataset["train"])

# Mostrar las primeras frases
df.head()


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/8.88k [00:00<?, ?B/s]

financial_phrasebank.py:   0%|          | 0.00/6.04k [00:00<?, ?B/s]

The repository for financial_phrasebank contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/financial_phrasebank.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] yes


FinancialPhraseBank-v1.0.zip:   0%|          | 0.00/682k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2264 [00:00<?, ? examples/s]

,sentence,label
0,"According to Gran , the company has no plans t...",1
1,"For the last quarter of 2010 , Componenta 's n...",2
2,"In the third quarter of 2010 , net sales incre...",2
3,Operating profit rose to EUR 13.1 mn from EUR ...,2
4,"Operating profit totalled EUR 21.1 mn , up fro...",2


### 4. Cargar el modelo FinBERT de Hugging Face y probar el modelo con algunas frases del dataset

In [6]:
model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Crear el pipeline de análisis de sentimiento
finbert_pipeline = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

# Probar el modelo con algunas frases del dataset

sample_phrases = df["sentence"][:10].tolist()
results = finbert_pipeline(sample_phrases)

# Combinar resultados con frases
for phrase, result in zip(sample_phrases, results):
    print(f"Frase: {phrase}\n→ Sentimiento: {result['label']} (score: {result['score']:.2f})\n")


Device set to use cpu


Frase: According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .
→ Sentimiento: neutral (score: 0.89)

Frase: For the last quarter of 2010 , Componenta 's net sales doubled to EUR131m from EUR76m for the same period a year earlier , while it moved to a zero pre-tax profit from a pre-tax loss of EUR7m .
→ Sentimiento: positive (score: 0.95)

Frase: In the third quarter of 2010 , net sales increased by 5.2 % to EUR 205.5 mn , and operating profit by 34.9 % to EUR 23.5 mn .
→ Sentimiento: positive (score: 0.96)

Frase: Operating profit rose to EUR 13.1 mn from EUR 8.7 mn in the corresponding period in 2007 representing 7.7 % of net sales .
→ Sentimiento: positive (score: 0.95)

Frase: Operating profit totalled EUR 21.1 mn , up from EUR 18.6 mn in 2007 , representing 9.7 % of net sales .
→ Sentimiento: positive (score: 0.96)

Frase: Finnish Talentum reports its operating profit increased to EUR 20.5 mn in 2005 from EUR 9

# parte 2

### 1. Instalar librerías necesarias e importar librerías


In [7]:
!pip install transformers pymupdf --quiet


import fitz  # PyMuPDF
from transformers import pipeline
import os




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 28.4 MB/s eta 0:00:00


### 3. Subir el PDF del contrato (se abrirá selector de archivos)

In [8]:
# Subir el pdf
from google.colab import files
uploaded = files.upload()

# Tomar el nombre del archivo cargado
pdf_file = next(iter(uploaded))



Saving Contrato_ejercicio.pdf to Contrato_ejercicio.pdf


### 4. Extraer texto del PDF

In [9]:
# Extraer texto
doc = fitz.open(pdf_file)
full_text = ""
for page in doc:
    full_text += page.get_text()

# Mostrar una parte del texto extraído (opcional)
print("🔹 Texto extraído (primeros 1000 caracteres):\n")
print(full_text[:1000])

🔹 Texto extraído (primeros 1000 caracteres):

ARRENDAMIENTO DE LOCAL COMERCIAL 
 
 
Conste por la suscripción del presente contrato de Arrendamiento o Alquiler del Local 
Comercial que celebran, de una parte ZAMORA QUISPE, MARÍA ELENA, 
identificada con DNI N.º 00000001, con domicilio real en Calle Los Cipreses N.º 123, 
Urb. Jardines del Sol, Distrito de San Martín de Porres, a quien en adelante se le 
denominará LA DUEÑA o ARRENDADORA; y de la otra parte, el señor 
VELÁSQUEZ MENDOZA, JULIO, identificado con DNI N.º 00000002, con domicilio 
en Mz. A Lt. 15, Conjunto Residencial Santa Clara, Distrito de Los Olivos, a quien 
en adelante se le denominará EL ARRENDATARIO o INQUILINO, bajo los 
siguientes términos: 
 
ANTECEDENTES 
 
 
PRIMERA: LA ARRENDADORA declara ser propietaria del inmueble ubicado en la 
Av. Los Laureles N.º 450, Mz. B Lt. 20, Urb. Villa Central, Distrito de 
Independencia, entre las avenidas Los Laureles y Alameda del Norte. 
 
 
SEGUNDA: LA ARRENDADORA deja constan

### 5. Crear el pipeline de resumen

In [10]:

from transformers import pipeline
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")




config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cpu


### 6. Aplicar el resumen (puedes limitar el texto si es muy largo)

In [11]:

summary = summarizer(full_text[:1024], max_length=200, min_length=30, do_sample=False)



### 7. Mostrar el resumen

In [12]:

print("🔸 Resumen generado:\n")
print(summary[0]['summary_text'])

🔸 Resumen generado:

Arrendamiento o Alquiler del Local                 Comercial que celebran, de una parte ZAMORA QUISPE, MARÍA ELENA, y de la otra parte, el señor VELÁSQUEZ MENDOZA.
